# Evaluation Analysis Demo

This notebook demonstrates corpus-level and sentence-level evaluation using BLEU, chrF, and linguistic error analysis.

In [13]:
from pathlib import Path
import os
import sys
import importlib

import pandas as pd

def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'config.py').exists() and (candidate / 'evaluation').exists():
            return candidate
    raise RuntimeError('Could not locate Machine_Translation project root.')

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

# Force reload to pick up local code edits in active notebook sessions.
import evaluation.linguistic_report as _linguistic_report
import evaluation.evaluation_pipeline as _evaluation_pipeline
importlib.reload(_linguistic_report)
importlib.reload(_evaluation_pipeline)
from evaluation.evaluation_pipeline import EvaluationPipeline, SentenceEvaluator

print(f'Project root: {PROJECT_ROOT}')
print('Reloaded evaluation modules.')

Project root: C:\Users\Nikolai\OneDrive\Desktop\Portfolio\CS_Language_Portfolio\projects\Machine_Translation
Reloaded evaluation modules.


In [14]:
sources = [
    'The company will not increase prices next year.',
    'Good morning, how are you?',
    'I have never seen this before.',
    'Machine translation is fascinating.',
    'The weather is beautiful today.'
]

hypotheses = [
    'Die Firma wird Preise erhoehen naechstes Jahr.',
    'Guten Morgen, wie geht es dir?',
    'Ich habe dieses gesehen zuvor nie.',
    'Maschinelle Uebersetzung ist faszinierend.',
    'Das Wetter ist heute schoen.'
]

references_list = [
    ['Die Firma wird die Preise naechstes Jahr nicht erhoehen.'],
    ['Guten Morgen, wie geht es Ihnen?'],
    ['Ich habe so etwas noch nie gesehen.'],
    ['Maschinelle Uebersetzung ist faszinierend.'],
    ['Das Wetter ist heute wunderbar.']
]

In [15]:
pipeline = EvaluationPipeline(bleu_max_n=4, include_chrf=True, include_linguistic=True)
result = pipeline.evaluate(
    hypotheses=hypotheses,
    references_list=references_list,
    source_lang='en',
    target_lang='de',
    sources=sources
)

summary = {
    'BLEU': result.bleu.score,
    'chrF': result.chrf.score if result.chrf else None,
    'BLEU_BP': result.bleu.bp,
    'BLEU_length_ratio': result.bleu.ratio
}

pd.DataFrame([summary]).round(4)

,BLEU,chrF,BLEU_BP,BLEU_length_ratio
0,0.4299,0.6364,0.8984,0.9032


In [16]:
if result.error_analysis is None:
    raise RuntimeError('Expected linguistic analysis results, but got None.')

error_dist = result.error_analysis.error_distribution()
quality_dist = result.error_analysis.quality_distribution()

error_df = pd.DataFrame(
    sorted(error_dist.items(), key=lambda x: x[1], reverse=True),
    columns=['error_category', 'count']
)

quality_df = pd.DataFrame(
    sorted(quality_dist.items(), key=lambda x: x[1], reverse=True),
    columns=['quality_label', 'count']
)

error_df, quality_df

(       error_category  count
 0      semantic_drift      7
 1    word_order_shift      5
 2    lexical_mismatch      2
 3  compound_breakdown      2,
   quality_label  count
 0          good      3
 1          poor      2)

In [17]:
evaluator = SentenceEvaluator()
single = evaluator.evaluate_sentence(
    source='The system should never modify user data without consent.',
    hypothesis='Das System sollte Benutzerdaten aendern ohne Zustimmung.',
    references=[
        'Das System sollte Benutzerdaten niemals ohne Zustimmung aendern.',
        'Das System darf Benutzerdaten ohne Zustimmung niemals veraendern.'
    ],
    source_lang='en',
    target_lang='de'
)

single

{'hypothesis': 'Das System sollte Benutzerdaten aendern ohne Zustimmung.',
 'references': ['Das System sollte Benutzerdaten niemals ohne Zustimmung aendern.',
  'Das System darf Benutzerdaten ohne Zustimmung niemals veraendern.'],
 'metrics': {'bleu': {'score': 0.3768499164492419,
   'precisions': [0.7142857142857143, 0.5, 0.4, 0.25]},
  'chrf': {'score': 0.70446735395189}},
 'linguistic_analysis': {'source': 'The system should never modify user data without consent.',
  'target': 'Das System sollte Benutzerdaten aendern ohne Zustimmung.',
  'reference': 'Das System sollte Benutzerdaten niemals ohne Zustimmung aendern.',
  'source_lang': 'en',
  'target_lang': 'de',
  'errors': [{'category': 'oov_untranslated',
    'source_span': 'system',
    'target_span': 'system',
    'reference_span': None,
    'severity': 'high',
    'explanation': "Word 'system' appears untranslated or OOV in target",
    'confidence': 0.8},
   {'category': 'lexical_mismatch',
    'source_span': 'The system shou

## Notes

- BLEU and chrF provide overlap-based quality signals.
- Linguistic analysis highlights interpretable failure types (negation, order, morphology).
- Use this notebook to compare model variants from the model selection demo.